[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/gregwchase/dynatrace-pubsec-summit-2026/blob/main/notebooks/02_evals.ipynb)

# Lab 2: Measure Performance with Evals
**Traces to Insights: A Hands-On Workshop with Arize AX** · Dynatrace Public Sector Summit 2026 · ⏱️ ~25 min

**In this lab you will:**
- Pull your traces out of Arize into a DataFrame
- Write **code-based evaluators** (fast, free, deterministic)
- Build **LLM-as-a-judge** evaluators for **hallucination** and **tool-call accuracy**
- Log eval results back to Arize so they're attached to every trace

In [ ]:
# --- Setup: works in Google Colab or a local Jupyter environment ---
import os, sys, subprocess

if "google.colab" in sys.modules:
    REPO_DIR = "/content/dynatrace-pubsec-summit-2026"
    if not os.path.exists(REPO_DIR):
        subprocess.run(["git", "clone", "-q", "https://github.com/gregwchase/dynatrace-pubsec-summit-2026.git", REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
elif not os.path.exists("civic_agent") and os.path.exists("../civic_agent"):
    os.chdir("..")  # running locally from notebooks/

sys.path.insert(0, os.getcwd())
from workshop_utils import load_credentials

PROJECT = load_credentials()  # prompts for keys if they aren't in .env / Colab Secrets

## 1. Export traces

We pull every span from the last 24 hours, then collapse each trace into one row: the resident's question, the final answer, which tools ran, and what those tools returned (`tool_context`).

In [ ]:
from workshop_utils import export_spans, root_spans

spans = export_spans(hours=24)
traces = root_spans(spans)
traces["tools_called_str"] = traces["tool_calls"].apply(lambda t: ", ".join(t) or "(none)")
traces[["input", "output", "tools_called_str", "guardrail_triggered"]].head(10)

## 2. Code-based evaluators

Use code when the rule is objective. These run in milliseconds and cost nothing, so you can run them on 100% of traffic.

1. **`cites_source`**: if tools returned a program record, does the answer include the official source link?
2. **`eligibility_tool_used`**: if the answer makes an eligibility claim ("you qualify", "you're eligible"), did the agent actually call `check_eligibility`? Or did it do the math itself?

In [ ]:
import re
from phoenix.evals import create_evaluator

@create_evaluator(name="cites_source", kind="code")
def cites_source(output: str, tool_context: str) -> float:
    returned_program = "source_url" in tool_context
    if not returned_program:
        return 1.0  # nothing to cite, so this is a pass
    return 1.0 if "dcs.columbia.example.gov" in output else 0.0

ELIGIBILITY_CLAIM = re.compile(r"\b(you|you're|you are|your household|you'd|we|they)\b.{0,40}\b(qualif|eligib)", re.I)

@create_evaluator(name="eligibility_tool_used", kind="code")
def eligibility_tool_used(output: str, tools_called_str: str) -> float:
    makes_claim = bool(ELIGIBILITY_CLAIM.search(output))
    if not makes_claim:
        return 1.0
    return 1.0 if "check_eligibility" in tools_called_str else 0.0

## 3. LLM-as-a-judge evaluators

Some qualities need judgment. We use a stronger model as the judge, give it a focused rubric, and ask it for a **label + explanation**. The explanation is what makes an eval useful for debugging.

### 3a. Hallucination: is the answer grounded in what the tools returned?

In [ ]:
from phoenix.evals import LLM, create_classifier

judge = LLM(provider="openai", model="gpt-4.1")

HALLUCINATION_TEMPLATE = """You are auditing a government benefits assistant. Its answers must be
based ONLY on information returned by its official tools, never on outside knowledge.

[Resident question]: {{input}}
[Tool results the assistant received]: {{tool_context}}
[Assistant answer]: {{output}}

Does the answer state any program, rule, amount, date, process, or recommendation that is
NOT supported by the tool results? General courtesy, a referral to the DCS hotline, or a
request for more information does not count as hallucination.

Respond "hallucinated" if the answer contains unsupported claims, otherwise "factual"."""

hallucination = create_classifier(
    name="hallucination",
    prompt_template=HALLUCINATION_TEMPLATE,
    llm=judge,
    choices={"factual": 1.0, "hallucinated": 0.0},
)

### 3b. Tool-call accuracy: did the agent pick the right tools for the question?

In [ ]:
TOOL_ACCURACY_TEMPLATE = """You are evaluating whether an AI agent used its tools correctly.

Available tools:
- search_programs(query): look up DCS program details. Required for ANY question about a program.
- check_eligibility(program_id, household_size, monthly_income): required for ANY eligibility
  determination. If household size or income is missing, the agent should ask instead.
- get_application_status(application_id): required for application status questions.
- find_office(zip_code): required for office location or hours questions.

[Resident question]: {{input}}
[Tools the agent called, in order]: {{tools_called_str}}
[Agent's final answer]: {{output}}

Respond "correct" if the agent called every tool the question required (and none that were
clearly wrong), or appropriately asked for missing information. Otherwise respond "incorrect"."""

tool_accuracy = create_classifier(
    name="tool_call_accuracy",
    prompt_template=TOOL_ACCURACY_TEMPLATE,
    llm=judge,
    choices={"correct": 1.0, "incorrect": 0.0},
)

## 4. Run all four evaluators

Guardrail-blocked traces never reached the LLM, so we skip them.

In [ ]:
from phoenix.evals import evaluate_dataframe

to_eval = traces[traces["guardrail_triggered"].fillna("none") == "none"].copy()
to_eval["output"] = to_eval["output"].fillna("")

results = evaluate_dataframe(
    dataframe=to_eval,
    evaluators=[cites_source, eligibility_tool_used, hallucination, tool_accuracy],
)

EVALS = ["cites_source", "eligibility_tool_used", "hallucination", "tool_call_accuracy"]
summary = {name: results[f"{name}_score"].apply(lambda s: s.get("score") if isinstance(s, dict) else None).mean() for name in EVALS}
print("Pass rate by evaluator (v1 prompt):")
for k, v in summary.items():
    print(f"  {k:<24} {v:.0%}")

Look at a few failures and read the judge's explanation:

In [ ]:
import pandas as pd
pd.set_option("display.max_colwidth", 200)

def failures(name, n=5):
    s = results[f"{name}_score"]
    mask = s.apply(lambda x: isinstance(x, dict) and x.get("score") == 0)
    out = results.loc[mask, ["input", "output"]].copy()
    out["explanation"] = s[mask].apply(lambda x: x.get("explanation"))
    return out.head(n)

failures("hallucination")

In [ ]:
failures("tool_call_accuracy")

## 5. Log evals back to Arize

Each eval is attached to the root span through `context.span_id`. In Arize you can then filter traces by eval label, chart pass rates on a dashboard, and add the failures to a dataset.

In [ ]:
from workshop_utils import to_arize_eval_columns, log_evals

eval_df = to_arize_eval_columns(results, EVALS)
log_evals(eval_df)
eval_df.head()

## 6. Explore in Arize 🔍

1. Open your project and refresh. You'll see new **eval columns** on the traces table.
2. Filter: `eval.hallucination.label = "hallucinated"`. These are your highest-risk traces.
3. Open one and read the explanation next to the trace.

### Bonus: run evals continuously in production
You ran these evals once, from a notebook. In production, you'd set up an **online eval task** in Arize (**Evals & Tasks → New Task**). Pick an evaluator, set a filter (e.g., root spans), set a sampling rate, and Arize scores new traces automatically as they arrive. That's how you **catch issues in production** instead of finding out from a complaint.

🤔 **Discuss:** which eval would you run on 100% of traffic, and which would you sample? Why?

**Next:** [Lab 3: Iterate with datasets and experiments →](03_datasets_experiments.ipynb)